In [1]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier


def build_id_path_map(root_dir: str) -> dict:
    """
    Walk through sub‑folders of `root_dir` (e.g. train/ or test/) and
    create a dictionary mapping each snippet id (filename without .npy)
    to its absolute file path.
    """
    mapping = {}
    pattern = os.path.join(root_dir, "*", "*.npy")
    for file_path in glob.glob(pattern):
        fname = os.path.basename(file_path)
        iid = fname[:-4]  # strip .npy
        mapping[iid] = file_path
    return mapping


def _read_csv_candidates(fname):
    """
    Try a handful of typical Kaggle locations for the CSV file.
    Returns a pandas DataFrame if found, otherwise raises FileNotFoundError.
    """
    candidates = [
        fname,
        os.path.join("input", fname),
        os.path.join("kaggle", "input", fname),
        os.path.join("kaggle", "input", "seti-breakthrough-listen", fname),
        os.path.join("kaggle", "input", "seti-breakthrough-listen", "data", fname),
        os.path.join("kaggle", "input", "seti-breakthrough-listen", "train", fname),
        os.path.join("kaggle", "input", "seti-breakthrough-listen", "test", fname),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f"Unable to locate {fname} in any known location.")


# Resolve train / test root directories
TRAIN_ROOT = "train"
TEST_ROOT = "test"

if not os.path.isdir(TRAIN_ROOT):
    TRAIN_ROOT = os.path.join("kaggle", "input", "seti-breakthrough-listen", "train")
if not os.path.isdir(TEST_ROOT):
    TEST_ROOT = os.path.join("kaggle", "input", "seti-breakthrough-listen", "test")

# Build id‑to‑path maps
id_to_path = build_id_path_map(TRAIN_ROOT)
test_id_to_path = build_id_path_map(TEST_ROOT)

# Load labels and sample submission using the robust helper
train_labels = _read_csv_candidates("train_labels.csv")
sample_submission = _read_csv_candidates("sample_submission.csv")
test_ids = sample_submission["id"].tolist()




FileNotFoundError: Unable to locate train_labels.csv in any known location.

In [2]:
def _compute_features_for_id(iid, id_path_map):
    """Compute an extended feature vector for a single id."""
    path = id_path_map.get(iid)
    if path and os.path.exists(path):
        arr = np.load(path, mmap_mode="r").astype(np.float32)  # (6, 273, 256)

        overall_mean = arr.mean()
        overall_std = arr.std()
        overall_var = arr.var()
        overall_max = arr.max()
        overall_min = arr.min()
        overall_median = np.median(arr)

        q25, q75 = np.quantile(arr, [0.25, 0.75])
        overall_iqr = q75 - q25
        overall_range = overall_max - overall_min

        pos_means = arr.mean(axis=(1, 2))
        pos_stds = arr.std(axis=(1, 2))
        pos_max = arr.max(axis=(1, 2))
        pos_min = arr.min(axis=(1, 2))
        pos_medians = np.median(arr, axis=(1, 2))

        on_idx = [0, 2, 4]
        off_idx = [1, 3, 5]

        on_mean = pos_means[on_idx].mean()
        off_mean = pos_means[off_idx].mean()
        diff_mean = on_mean - off_mean
        ratio_mean = on_mean / (off_mean + 1e-6)

        on_std = pos_stds[on_idx].mean()
        off_std = pos_stds[off_idx].mean()
        diff_std = on_std - off_std
        ratio_std = on_std / (off_std + 1e-6)

        on_max = pos_max[on_idx].mean()
        off_max = pos_max[off_idx].mean()
        diff_max = on_max - off_max
        ratio_max = on_max / (off_max + 1e-6)

        on_min = pos_min[on_idx].mean()
        off_min = pos_min[off_idx].mean()
        diff_min = on_min - off_min
        ratio_min = on_min / (off_min + 1e-6)

        var_pos_means = pos_means.var()
        var_pos_stds = pos_stds.var()
        var_pos_max = pos_max.var()
        var_pos_min = pos_min.var()

        return np.concatenate(
            [
                [
                    overall_mean,
                    overall_std,
                    overall_var,
                    overall_max,
                    overall_min,
                    overall_median,
                    overall_range,
                    overall_iqr,
                ],
                pos_means,
                pos_stds,
                pos_max,
                pos_min,
                pos_medians,
                [
                    diff_mean,
                    diff_std,
                    diff_max,
                    diff_min,
                    ratio_mean,
                    ratio_std,
                    ratio_max,
                    ratio_min,
                ],
                [
                    var_pos_means,
                    var_pos_stds,
                    var_pos_max,
                    var_pos_min,
                ],
            ]
        ).astype(np.float32)
    else:
        # Return a vector of NaNs that will be imputed later
        return np.full(50, np.nan, dtype=np.float32)


def extract_features(ids, id_path_map):
    """Parallel feature extraction (process‑based) preserving order."""
    import concurrent.futures

    max_workers = os.cpu_count() or 1
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(
            _compute_features_for_id, ids, [id_path_map] * len(ids), chunksize=64
        )
        feature_array = np.vstack(list(results))
    return feature_array




In [3]:
# ----- training pipeline -----
train_ids = train_labels["id"].tolist()
raw_train_feats = extract_features(train_ids, id_to_path)

imputer = SimpleImputer(strategy="mean")
train_feats = imputer.fit_transform(raw_train_feats)

train_targets = train_labels.set_index("id").loc[train_ids, "target"].values

scaler = StandardScaler()
train_feats_scaled = scaler.fit_transform(train_feats)

model = ExtraTreesClassifier(
    n_estimators=1500,  # slightly more trees for a modest boost
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
    min_samples_leaf=1,
    max_features="sqrt",
)
model.fit(train_feats_scaled, train_targets)




NameError: name 'train_labels' is not defined

In [4]:
# ----- inference on test set -----
raw_test_feats = extract_features(test_ids, test_id_to_path)

test_feats = imputer.transform(raw_test_feats)
test_feats_scaled = scaler.transform(test_feats)

test_probs = model.predict_proba(test_feats_scaled)[:, 1]

submission = pd.DataFrame({"id": test_ids, "target": test_probs})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

NameError: name 'test_ids' is not defined